# Workshop 2 — Embedding-based RAG with NVIDIA NIM

**Picks up where [Workshop 1](https://github.com/torkian/nvidia-nim-workshop/blob/main/notebook.ipynb) left off.** In Workshop 1 we pasted the whole knowledge base into the prompt. Here we replace that with real retrieval — embed once, embed the query, score with cosine similarity, send only the top chunks to the LLM.

No vector database needed for the basics. A Python list + NumPy is enough to understand what's actually happening.

## Step 1 — Setup (same as Workshop 1)

Paste your NVIDIA API key when prompted. Skip this cell if you're continuing from Workshop 1 in the same Colab session.

In [ ]:
%pip install -q openai numpy

import os, getpass
from openai import OpenAI
import numpy as np

if not os.getenv('NVIDIA_API_KEY'):
    os.environ['NVIDIA_API_KEY'] = getpass.getpass('Paste your NVIDIA API key (starts with nvapi-): ')

client = OpenAI(
    base_url='https://integrate.api.nvidia.com/v1',
    api_key=os.environ['NVIDIA_API_KEY'],
)

MODEL = 'meta/llama-3.1-8b-instruct'
EMBED_MODEL = 'nvidia/nv-embedqa-e5-v5'

def ask(system_prompt, user_message):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ],
        temperature=0.3,
        max_tokens=400,
    )
    return response.choices[0].message.content

print('Ready. Chat model:', MODEL, ' | Embed model:', EMBED_MODEL)

## Step 2 — Build a knowledge base and embed it as passages

**Key NVIDIA-specific detail:** `nv-embedqa-e5-v5` treats stored documents (`passage`) and search questions (`query`) differently. You pass `input_type` via `extra_body`.

In [ ]:
knowledge_base = [
    {'title': 'USC AI Club meeting',
     'text': 'The USC AI Club meets every Thursday at 5 PM in the engineering building, room 204.'},
    {'title': 'USC GPU lab hours',
     'text': 'The USC GPU computing lab is open Monday to Friday from 10 AM to 6 PM.'},
    {'title': 'NVIDIA Developer Program',
     'text': 'USC students can join the NVIDIA Developer Program for free.'},
    {'title': 'Next USC workshop',
     'text': 'The next USC AI Club workshop will cover Retrieval Augmented Generation (RAG).'},
    {'title': 'USC AI/ML office hours',
     'text': 'Office hours for the USC AI/ML faculty are Tuesdays 2-4 PM.'},
    {'title': 'USC robotics lab',
     'text': 'The USC robotics lab requires safety training before students can use the soldering station.'},
    {'title': 'USC tutoring',
     'text': 'Peer tutoring for introductory Python at USC is available Wednesdays from 1 PM to 3 PM.'},
]

def embed_texts(texts, input_type='passage'):
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts,
        extra_body={'input_type': input_type},
    )
    return [np.array(item.embedding, dtype=np.float32) for item in response.data]

embeddings = embed_texts([item['text'] for item in knowledge_base], input_type='passage')
for item, embedding in zip(knowledge_base, embeddings):
    item['embedding'] = embedding

print(f'Embedded {len(knowledge_base)} chunks. Vector dim:', embeddings[0].shape[0])

## Step 3 — Retrieve the top-k chunks for a question

User questions go in as `query`, not `passage`. Same model, different mode.

In [ ]:
def cosine_similarity(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

def retrieve_context(question, k=3):
    question_embedding = embed_texts([question], input_type='query')[0]
    scored = []
    for item in knowledge_base:
        score = cosine_similarity(question_embedding, item['embedding'])
        scored.append((score, item))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    top_items = [item for score, item in scored[:k]]
    return '\n'.join(f"- {item['text']}" for item in top_items)

## Step 4 — Retrieval-augmented `ask()`

Same `ask()` from Workshop 1, but `{context}` now comes from `retrieve_context()` instead of being hardcoded.

In [ ]:
def ask_with_retrieval(question):
    context = retrieve_context(question)
    system_prompt = f'''You are a USC campus assistant. Answer ONLY using the
context below. If the answer is not in the context, say
"I don't have that information — check with the USC AI Club."

CONTEXT:
{context}
'''
    return ask(system_prompt, question)

for question in [
    'Where does the USC AI Club meet?',
    'When can I get Python tutoring at USC?',
    'What is the wifi password?',
]:
    print(f'Q: {question}')
    print(f'Context:\n{retrieve_context(question)}')
    print(f'A: {ask_with_retrieval(question)}\n')

## What just happened

- **Q1 (AI Club):** the AI Club chunk ranked first, model answered from it.
- **Q2 (Python tutoring):** semantic match worked — your query said "Python tutoring," the chunk says "introductory Python." Embeddings know those are close.
- **Q3 (wifi):** none of the top-k chunks contain a password. The guardrail from Workshop 1 fires and the assistant refuses instead of inventing one.

## Where this goes next

Workshop 3 adds a second guardrail layer: a tiny verifier call that re-reads the answer and the context, and refuses if anything in the answer isn't supported. Plus prompt-scope rules so off-topic questions get the same refusal.

Replace `knowledge_base` with your own data and run it before moving on. Star the repo: https://github.com/torkian/nvidia-nim-workshop